In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../'))

import numpy as np
from wiener_chaos.pricing.heston import heston_price, calibrate_heston_to_iv_market
from wiener_chaos.pricing.options import implied_vol_bisection
import pickle
import torch

In [ ]:
# Model + market setup
S0 = 100.0
r = 0.0
q = 0.0
params = (1.5, 0.04, 0.5, -0.7, 0.04)   # (kappa, theta, sigma, rho, v0)

# "Wise" grids: avoid exactly T=0; cover short to 1y; 0.8–1.2 moneyness
maturities_calibr = [0.0821, 0.1725,0.2491, 0.4983, 0.9884, 1.4867, 1.974]   
strikes_calibr = [np.linspace(0.8*S0, 1.2*S0, 9).tolist() for _ in range(len(maturities_calibr))]

maturities_test = [0.13, 0.21, 0.35, 0.75, 1.25, 1.75]
strikes_test = [np.linspace(0.8*S0, 1.2*S0, 9).tolist() for _ in range(len(maturities_test))]

In [ ]:
prices_calibr = []
iv_calibr = []

for t, T in enumerate(maturities_calibr):
    prices_t, iv_t = [], []
    for k, K in enumerate(strikes_calibr[t]):
        c = heston_price(params, r, q, T, S0, K, flag="call")
        iv_c = implied_vol_bisection(c, S0, K, T, r, q, flag="call")
        prices_t.append(c)
        iv_t.append(iv_c)
    prices_calibr.append(prices_t)
    iv_calibr.append(iv_t)

In [ ]:
prices_test = []
iv_test = []

for t, T in enumerate(maturities_test):
    prices_t, iv_t = [], []
    for k, K in enumerate(strikes_test[t]):
        c = heston_price(params, r, q, T, S0, K, flag="call")
        iv_c = implied_vol_bisection(c, S0, K, T, r, q, flag="call")
        prices_t.append(c)
        iv_t.append(iv_c)
    prices_test.append(prices_t)
    iv_test.append(iv_t)

In [ ]:
heston_data = {}
heston_data["params"] = params

heston_data["maturities_calibr"] = maturities_calibr
heston_data["strikes_calibr"] = strikes_calibr
heston_data["prices_calibr"] = prices_calibr
heston_data["iv_calibr"] = iv_calibr

heston_data["maturities_test"] = maturities_test
heston_data["strikes_test"] = strikes_test
heston_data["prices_test"] = prices_test
heston_data["iv_test"] = iv_test

heston_data["r"] = r
heston_data["q"] = q
heston_data["S0"] = S0

with open("heston_data.pkl", "wb") as f:
    pickle.dump(heston_data, f)

In [ ]:
# ---- Plot Heston IV surface: x=strikes, y=maturities ----
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed for 3D projection)

# Build mesh so shapes match iv/prices: (len(strikes), len(maturities))
X, Y = np.meshgrid(np.array(strikes_calibr[0]), np.array(maturities_calibr), indexing="ij")  # X[i,j]=strike_i, Y[i,j]=mat_j

Z = np.array(iv_calibr).T  # use 'prices' here instead if you want the price surface

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

surf = ax.plot_surface(X, Y, Z, cmap="viridis", edgecolor="none")

ax.set_xlabel("Strike")
ax.set_ylabel("Maturity")
ax.set_zlabel("Implied Volatility")  # change to "Price" if Z=prices
ax.set_title("Heston Implied Volatility Surface")

cbar = fig.colorbar(surf, shrink=0.6, aspect=12)
cbar.set_label("IV")  # change to "Price" if Z=prices

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
for j, T in enumerate(maturities_calibr):
    plt.plot(
        strikes_calibr[j], np.array(iv_calibr[j]),
        marker="o", linewidth=1.8, label=f"T = {T:.2f}"
    )

plt.xlabel("Strike")
plt.ylabel("Implied Volatility")
plt.title("Heston IV Smiles by Maturity")
plt.legend(title="Maturity")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()